# Checkpoint IV · Examen blanc — notebook de l'examen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_4/02_examen_notebook.ipynb)

Ce notebook accompagne le sujet `01_examen_sujet.md`.

- **Partie A, pendant l'examen** : la question de code CP4.9 (🔨, 2 points, avec NumPy seulement). Écris ton code dans la cellule à compléter, sans `mylearn`, sans PyTorch, sans les corrigés et sans assistant IA ; `help()` et la documentation officielle de NumPy sont permis. Rien n'est vérifié pendant l'examen : la cellule « essai » montre seulement ce que fait ton code. Tes réponses chiffrées et tes explications vont sur ta feuille (`04_mes_reponses.md`).
- **Partie B, après l'examen** : la vérification automatique de ton code et des réponses de ta feuille. Ses cellules ne vérifient rien tant que `EXAM_OVER` vaut `False`.

« Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳ (partie A) ou ⏸️ (partie B).

> Travaille dans **ta copie** (`mon_travail/checkpoints/partie_4/02_examen_notebook.ipynb`, créée par `python tools/start_chapter.py CP4`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Pendant l'examen : la question de code

In [ ]:
# Data and tools of the code question CP4.9 (run this cell first)
import numpy as np

# 2 048 images of Fashion-MNIST, flattened to 784 values in [0, 1] (float64), and their labels 0-9
X_F, Y_F = wb.datasets.load_fashion_mnist("train", n=2048, flatten=True, normalize=True, seed=49)
X_F = X_F.astype(np.float64)
LR, BATCH, MAX_NORM = 1e-2, 100, 1.0          # the settings of the try (CP4.9 d)


def init_params(seed=49):
    """The starting parameters [W, b] of the softmax regression 784 -> 10 (new arrays at each call)."""
    rng = np.random.default_rng(seed)
    return [rng.normal(0.0, 0.01, size=(784, 10)), np.zeros(10)]


def loss_and_grads(params, X, y):
    """(mean cross-entropy of the mini-batch, [dW, db]) of the softmax regression z = X @ W + b."""
    W, b = params
    z = X @ W + b
    z = z - z.max(axis=1, keepdims=True)                       # log-sum-exp trick (ch. 17)
    p = np.exp(z)
    p /= p.sum(axis=1, keepdims=True)
    n = len(y)
    loss = -np.log(p[np.arange(n), y]).mean()
    d = p.copy()
    d[np.arange(n), y] -= 1.0                                  # dL/dz = (p - one_hot(y)) / n  (ch. 18)
    d /= n
    return float(loss), [X.T @ d, d.sum(axis=0)]


def accuracy(params, X, y):
    """Share of the images whose largest logit is the right class."""
    return float(((X @ params[0] + params[1]).argmax(axis=1) == y).mean())


_ORIGINALS = {"X_F": X_F.copy(), "Y_F": Y_F.copy()}         # part B restores them if a try changed them
print("X_F:", X_F.shape, "· Y_F:", Y_F.shape, "· starting loss:", round(loss_and_grads(init_params(), X_F, Y_F)[0], 4))

### CP4.9 — Écrire de mémoire une epoch d'entraînement avec AdamW et clipping, en NumPy 🔨 ★★ ⏱️ 15 min · 2 points
**Objectif :** écrire sans aide le cœur d'une boucle d'entraînement : clipping par la norme globale, un pas d'AdamW, une epoch en mini-batches.

Avec NumPy seulement. Une régression softmax (784 → 10) sur les 2 048 images de `X_F` et leurs labels `Y_F` ; ses paramètres sont la liste `params = [W, b]` (`init_params()` en donne une nouvelle à chaque appel). `loss_and_grads(params, X, y)` rend la cross-entropy moyenne du mini-batch et la liste `[dW, db]` de ses gradients ; `accuracy(params, X, y)` rend l'accuracy.

a) `clip_by_global_norm(grads, max_norm)` rend `(clipped, norm)` : `norm`, la norme globale (racine de la somme des carrés de **tous** les gradients) ; `clipped`, une liste de **nouveaux** tableaux, les gradients multipliés par $\min\left(1, \frac{\text{max\_norm}}{\text{norm} + 10^{-6}}\right)$. Elle ne modifie pas `grads`. **[0,5]**
b) `adamw_step(params, grads, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)` fait un pas d'AdamW **sur place**, avec les conventions de `torch.optim.AdamW` : $t \leftarrow t + 1$ ; $w \leftarrow w\,(1 - \eta\lambda)$ ; $m \leftarrow \beta_1 m + (1 - \beta_1)\,g$ ; $v \leftarrow \beta_2 v + (1 - \beta_2)\,g^2$ ; $w \leftarrow w - \eta\,\frac{m / (1 - \beta_1^t)}{\sqrt{v / (1 - \beta_2^t)} + \epsilon}$. `state`, un dictionnaire vide au premier appel, garde ce qui sert d'un pas à l'autre. **[0,7]**
c) `train_epoch(params, X, y, state, lr, batch_size, max_norm, rng)` fait une epoch : `order = rng.permutation(len(X))`, puis les mini-batches de `batch_size` exemples dans cet ordre (le dernier peut être plus petit) ; pour chacun, `loss_and_grads`, le clipping, puis `adamw_step` (les autres réglages par défaut). Elle rend la loss moyenne de l'epoch par **exemple** (chaque loss de mini-batch pesée par son nombre d'exemples), un `float`. **[0,5]**
d) La cellule d'essai entraîne une epoch avec `LR`, `BATCH` et `MAX_NORM`, et `np.random.default_rng(49)` : reporte sur ta feuille $[\text{loss}, \text{accuracy}]$ (4 et 3 décimales). **[0,3]**

In [ ]:
def clip_by_global_norm(grads, max_norm):
    """(clipped, norm): new arrays scaled by min(1, max_norm / (norm + 1e-6)), and the global norm; CP4.9 a)."""
    raise NotImplementedError("clip_by_global_norm")  # TODO CP4.9 a)


def adamw_step(params, grads, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW step, in place on the arrays of params, like torch.optim.AdamW; state is a dict; CP4.9 b)."""
    raise NotImplementedError("adamw_step")  # TODO CP4.9 b)


def train_epoch(params, X, y, state, lr, batch_size, max_norm, rng):
    """One epoch in the order rng.permutation(len(X)); returns the mean loss per example, a float; CP4.9 c)."""
    raise NotImplementedError("train_epoch")  # TODO CP4.9 c)

In [ ]:
# Try your functions (no check during the exam: compare with what you expect)
with wb.attempt("CP4.9"):
    clipped, norm = clip_by_global_norm([np.array([3.0, 4.0]), np.array([12.0])], 6.5)
    print("clip of [3, 4], [12] at 6.5:", [c.round(4).tolist() for c in clipped], "· norm", norm)
with wb.attempt("CP4.9"):
    params_try, state_try = init_params(), {}
    loss_try = train_epoch(params_try, X_F, Y_F, state_try, LR, BATCH, MAX_NORM, np.random.default_rng(49))
    print(f"one epoch: mean loss {loss_try:.4f} · accuracy {accuracy(params_try, X_F, Y_F):.3f}")

---

### 🛑 Fin de la partie A

L'examen s'arrête ici. Rends ta copie (enregistre ce notebook et ta feuille), puis, seulement après, passe à la partie B.

## Partie B · Après l'examen : vérification automatique

**Ne commence cette partie qu'une fois ta copie terminée**, examen rendu. Elle vérifie le code de la partie A, puis les réponses que tu as écrites sur ta feuille, question par question : ✅ juste, ❌ faux (avec une piste), ⏳ pas rempli. Elle ne note pas : les démarches, les justifications et les questions rédigées se corrigent avec `03_examen_corrige.md`, qui donne aussi le barème.

1. Passe `EXAM_OVER` à `True` dans la cellule ci-dessous, puis exécute-la.
2. Exécute la vérification du code (CP4.9).
3. Reporte ensuite chaque réponse de ta feuille : **la valeur** que tu as écrite, pas un nouveau calcul (sinon tu ne vérifies plus ta copie). En Python, le séparateur décimal est un **point** (`0.125`), une matrice s'écrit comme une liste de lignes (`[[1, 2], [3, 4]]`), et les lettres s'écrivent entre guillemets (`"XYZ"`).

Si tu as fermé le notebook depuis l'examen, exécute d'abord la cellule de configuration (tout en haut) et les cellules de la partie A, sauf les essais.

In [ ]:
EXAM_OVER = False   # set it to True once your exam is finished, then run the cells of part B

import re

RESULTS = {}        # sub-ID -> result of wb.check
PROPERTIES = {}     # question -> [(property, passed)], reset each time its check cell runs
_REMINDED = []      # the full reminder is printed once, then a short one


def exam_over():
    """True once the exam is finished; otherwise print a reminder (and the cell checks nothing)."""
    if not EXAM_OVER:
        print("⏸️ Rien n'est vérifié (EXAM_OVER vaut False)." if _REMINDED else
              "⏸️ Vérification après l'examen : passe EXAM_OVER à True dans la première cellule de la partie B.")
        _REMINDED.append(True)
        return False
    if "wb" not in globals():
        print("⚠️ Exécute d'abord la cellule de configuration, tout en haut du notebook.")
        return False
    return True


def ready(question, *names):
    """True when the cells of part A that define `names` have run (after a kernel restart, run them again)."""
    missing = [name for name in names if name not in globals()]
    if missing:
        verb = "n'existe" if len(missing) == 1 else "n'existent"
        print(f"⚠️ {question} : exécute d'abord les cellules de la partie A ({', '.join(missing)} {verb} pas encore).")
    return not missing


def verdict(question, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message, and remember the result."""
    PROPERTIES.setdefault(question, []).append((success or failure, bool(ok)))
    print(f"✅ {question} : {success}" if ok else f"❌ {question} : {failure}")
    return bool(ok)


_LIBRARY_ADVICE = " ; si elle fait partie de `mylearn`, ses tests (`pytest`) disent quel cas pose problème."


def code_check(ex_id, value, reminder=""):
    """wb.check of a value computed by your exam function (no advice about mylearn: it is not used here)."""
    result = wb.check(ex_id, value, computed=True, quiet=True)
    message = result.message.replace(_LIBRARY_ADVICE, ".")
    if not result and reminder and not message.startswith("Erreur classique"):
        message += f" Rappel : {reminder}"
    print(f"{'✅' if result else '❌'} Ex {ex_id} : {message}")
    RESULTS[ex_id] = result
    return result


def as_letters(value):
    """Letters typed as "ACE", "A, C et E" or ["A", "C", "E"] -> "ACE" (`...` stays `...`)."""
    if value is ... or value is None:
        return value
    if isinstance(value, (list, tuple)):
        value = " ".join(str(v) for v in value)
    value = re.sub(r"(?i)\b(et|and|ou|or)\b", " ", str(value))
    return "".join(ch for ch in value if ch.isalpha()).upper()


def as_sorted_letters(value):
    """A set of letters typed in any order -> its letters in alphabetical order."""
    letters = as_letters(value)
    return "".join(sorted(letters)) if isinstance(letters, str) else letters


def as_shape(value):
    """A shape typed as "(16, 32)", "[16, 32]", "16, 32", torch.Size([16, 32]) or a tuple -> "(16, 32)"."""
    if isinstance(value, (tuple, list)) and all(isinstance(v, int) and not isinstance(v, bool) for v in value):
        return str(tuple(value))
    if isinstance(value, str) and re.fullmatch(r"\s*(torch\.Size)?[\s(\[]*\d+(\s*,\s*\d+)*\s*,?[\s)\]]*\s*", value):
        return str(tuple(int(n) for n in re.findall(r"\d+", value)))
    return value


def restore_data(question, names):
    """Put back the data of part A if a try of part A changed them in place (a function that writes into its input)."""
    changed = [name for name in names
               if name not in globals() or not np.array_equal(globals()[name], _ORIGINALS[name])]
    for name in changed:
        globals()[name] = _ORIGINALS[name].copy()
    if changed:
        print(f"⚠️ {question} : {', '.join(changed)} avai{'en' if len(changed) > 1 else ''}t changé depuis la cellule des "
              f"données (une fonction de la partie A écrit-elle dans ce qu'elle reçoit ?) : la vérification repart "
              f"des données d'origine.")

### Le code de la partie A

In [ ]:
# Checks of clip_by_global_norm, adamw_step and train_epoch (CP4.9)
if exam_over() and ready("CP4.9", "np", "X_F", "Y_F", "init_params", "loss_and_grads", "accuracy", "_ORIGINALS",
                         "clip_by_global_norm", "adamw_step", "train_epoch"):
    PROPERTIES["CP4.9"] = []
    RESULTS.pop("CP4.9d", None)
    restore_data("CP4.9", ["X_F", "Y_F"])

    def oracle_step(params, grads, state, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        """torch.optim.AdamW transcribed in NumPy (its pseudo-code, PyTorch 2.11), in place."""
        if not state:
            state.update(t=0, m=[np.zeros_like(p) for p in params], v=[np.zeros_like(p) for p in params])
        state["t"] += 1
        for p, g, m, v in zip(params, grads, state["m"], state["v"]):
            p *= 1 - lr * weight_decay
            m[...] = betas[0] * m + (1 - betas[0]) * g
            v[...] = betas[1] * v + (1 - betas[1]) * g * g
            p -= lr * (m / (1 - betas[0] ** state["t"])) / (np.sqrt(v / (1 - betas[1] ** state["t"])) + eps)

    def safe(function, *args, **kwargs):
        """(True, result) or (False, the text of the error): a bug of the exam code must not stop part B."""
        try:
            return True, function(*args, **kwargs)
        except NotImplementedError:
            return None, None
        except Exception as exc:
            return False, f"{type(exc).__name__}: {exc}"

    # a) clip_by_global_norm
    grads_a = [np.array([3.0, 4.0]), np.array([12.0])]
    ok, out = safe(clip_by_global_norm, grads_a, 6.5)
    if ok is None:
        print("⏳ CP4.9 a) : clip_by_global_norm n'est pas encore écrite.")
    elif not ok:
        verdict("CP4.9", False, "", f"clip_by_global_norm lève une erreur : {out}")
    elif out is None:
        verdict("CP4.9", False, "", "clip_by_global_norm ne renvoie rien : as-tu oublié le return ?")
    elif not (isinstance(out, (tuple, list)) and len(out) == 2):
        verdict("CP4.9", False, "", f"clip_by_global_norm doit renvoyer le couple (clipped, norm), pas {out!r}"[:300])
    else:
        clipped, norm = out
        verdict("CP4.9", np.array_equal(grads_a[0], [3.0, 4.0]) and np.array_equal(grads_a[1], [12.0]),
                "clip_by_global_norm ne modifie pas les gradients qu'elle reçoit.",
                "clip_by_global_norm modifie grads (par exemple avec *=) : elle doit rendre de nouveaux tableaux.")
        try:
            norm_ok = abs(float(norm) - 13.0) < 1e-9
        except (TypeError, ValueError):
            norm_ok = False
        verdict("CP4.9", norm_ok, "la norme globale de [3, 4] et [12] vaut 13.",
                f"la norme rendue vaut {norm!r} : la norme GLOBALE est la racine de la somme des carrés de tous les "
                "gradients (3² + 4² + 12² = 169, donc 13), pas une norme par tableau.")
        coef = 6.5 / (13 + 1e-6)
        try:
            values_ok = (len(clipped) == 2 and np.allclose(clipped[0], [3 * coef, 4 * coef], rtol=1e-9, atol=0)
                         and np.allclose(clipped[1], [12 * coef], rtol=1e-9, atol=0))
        except (TypeError, ValueError):
            values_ok = False
        verdict("CP4.9", values_ok, "au-dessus de max_norm, les gradients sont multipliés par max_norm / (norm + 1e-6).",
                "avec max_norm = 6,5 et une norme de 13, chaque gradient doit être multiplié par 6,5 / (13 + 1e-6), "
                f"le même coefficient pour tous ; reçu {clipped!r}"[:300])
        ok_low, low = safe(clip_by_global_norm, [np.array([3.0, 4.0]), np.array([12.0])], 20.0)
        verdict("CP4.9", ok_low and isinstance(low, (tuple, list)) and len(low) == 2
                and np.array_equal(low[0][0], [3.0, 4.0]) and np.array_equal(low[0][1], [12.0]),
                "sous max_norm, les gradients ne changent pas (coefficient 1).",
                "avec max_norm = 20 > 13, les gradients ne doivent pas changer : le coefficient est min(1, ...).")

    # b) adamw_step against the transcription of torch.optim.AdamW
    rng_b = np.random.default_rng(491)
    start_b = [rng_b.normal(size=(3, 2)), rng_b.normal(size=2)]
    grads_b = [[rng_b.normal(size=(3, 2)), rng_b.normal(size=2)] for _ in range(3)]
    mine_b, want_b, state_mine, state_want = [p.copy() for p in start_b], [p.copy() for p in start_b], {}, {}
    objects_b = [id(p) for p in mine_b]
    ok, out = True, None
    for step_b, g in enumerate(grads_b):
        ok, out = safe(adamw_step, mine_b, [x.copy() for x in g], state_mine, 0.01, betas=(0.8, 0.95), eps=1e-6,
                       weight_decay=0.1)
        oracle_step(want_b, [x.copy() for x in g], state_want, 0.01, betas=(0.8, 0.95), eps=1e-6, weight_decay=0.1)
        if not ok:
            break
    if ok is None:
        print("⏳ CP4.9 b) : adamw_step n'est pas encore écrite.")
    elif not ok:
        verdict("CP4.9", False, "", f"adamw_step lève une erreur : {out}")
    else:
        verdict("CP4.9", [id(p) for p in mine_b] == objects_b and not np.array_equal(mine_b[0], start_b[0]),
                "adamw_step modifie les paramètres sur place.",
                "les tableaux de params n'ont pas changé : adamw_step doit les modifier SUR PLACE (p -= ..., p *= ...), "
                "pas créer de nouveaux tableaux.")
        gap_b = max(np.abs(a - b).max() for a, b in zip(mine_b, want_b))
        verdict("CP4.9", gap_b < 1e-10,
                "trois pas d'AdamW (betas, eps et weight_decay reçus en arguments) : les valeurs de torch.optim.AdamW.",
                f"après trois pas (lr 0,01, betas (0,8 ; 0,95), eps 1e-6, weight_decay 0,1), l'écart à torch.optim.AdamW "
                f"vaut {gap_b:.2e} : vérifie le compteur t (incrémenté avant de corriger le biais), le weight decay "
                "découplé (w × (1 − ηλ) avant le pas), la correction de biais, et que tu utilises les betas et eps reçus.")
        frozen = [np.array([2.0, -4.0])]
        ok_d, out_d = safe(adamw_step, frozen, [np.zeros(2)], {}, 0.1, weight_decay=0.5)
        verdict("CP4.9", ok_d and np.allclose(frozen[0], [1.9, -3.8], rtol=1e-12, atol=0),
                "avec un gradient nul, seul le weight decay agit : w × (1 − ηλ).",
                f"avec un gradient nul, lr 0,1 et weight_decay 0,5, w = [2, −4] doit devenir [1,9 ; −3,8] (w × (1 − ηλ), "
                f"le pas d'Adam étant nul) ; reçu {frozen[0]!r}.")

    # c) train_epoch against the same epoch with the oracle
    params_c = init_params()
    ok, loss_c = safe(train_epoch, params_c, X_F.copy(), Y_F.copy(), {}, LR, BATCH, MAX_NORM, np.random.default_rng(49))
    if ok is None:
        print("⏳ CP4.9 c) : train_epoch (ou une fonction qu'elle appelle) n'est pas encore écrite.")
    elif not ok:
        verdict("CP4.9", False, "", f"train_epoch lève une erreur : {loss_c}")
    elif loss_c is None:
        verdict("CP4.9", False, "", "train_epoch ne renvoie rien : as-tu oublié le return ?")
    else:
        want_c, state_c = init_params(), {}
        order_c = np.random.default_rng(49).permutation(len(X_F))
        total_c = 0.0
        for start in range(0, len(X_F), BATCH):
            idx = order_c[start:start + BATCH]
            loss_i, grads_i = loss_and_grads(want_c, X_F[idx], Y_F[idx])
            norm_i = np.sqrt(sum(np.sum(g ** 2) for g in grads_i))
            oracle_step(want_c, [g * min(1.0, MAX_NORM / (norm_i + 1e-6)) for g in grads_i], state_c, LR)
            total_c += loss_i * len(idx)
        gap_c = max(np.abs(a - b).max() for a, b in zip(params_c, want_c))
        verdict("CP4.9", isinstance(loss_c, float), "train_epoch rend un float.",
                f"train_epoch rend {type(loss_c).__name__} : attendu un float Python (return float(...)).")
        verdict("CP4.9", gap_c < 1e-10,
                "après l'epoch, les paramètres sont ceux de torch.optim.AdamW et clip_grad_norm_ sur les mêmes mini-batches.",
                f"après l'epoch, l'écart aux paramètres attendus vaut {gap_c:.2e} : l'ordre rng.permutation(len(X)), des "
                "mini-batches consécutifs de batch_size, le clipping de CHAQUE gradient avant le pas, un seul state pour "
                "toute l'epoch, et les réglages par défaut d'adamw_step.")
        try:
            weighted_ok = abs(float(loss_c) - total_c / len(X_F)) < 1e-9
        except (TypeError, ValueError):
            weighted_ok = False
        verdict("CP4.9", weighted_ok, "la loss rendue est la moyenne par exemple (le dernier mini-batch, de 48 images, pèse 48).",
                f"train_epoch rend {loss_c!r} ; la moyenne par exemple vaut {total_c / len(X_F):.6f} : pèse chaque loss de "
                "mini-batch par son nombre d'exemples.")
        try:
            import torch                                    # the same check against PyTorch itself, when installed
            W_t, b_t = (torch.tensor(a, requires_grad=True) for a in init_params())
            opt_t = torch.optim.AdamW([W_t, b_t], lr=LR, weight_decay=0.01)
            X_t, Y_t = torch.from_numpy(X_F), torch.from_numpy(Y_F)
            for start in range(0, len(X_F), BATCH):
                idx = torch.from_numpy(order_c[start:start + BATCH])
                loss_t = torch.nn.functional.cross_entropy(X_t[idx] @ W_t + b_t, Y_t[idx])
                opt_t.zero_grad()
                loss_t.backward()
                torch.nn.utils.clip_grad_norm_([W_t, b_t], MAX_NORM)
                opt_t.step()
            gap_t = max(np.abs(W_t.detach().numpy() - params_c[0]).max(), np.abs(b_t.detach().numpy() - params_c[1]).max())
            print(f"   (PyTorch : écart de tes paramètres à torch.optim.AdamW + clip_grad_norm_ : {gap_t:.1e})")
        except ImportError:
            pass
        code_check("CP4.9d", [loss_c, accuracy(params_c, X_F, Y_F)],
                   "une epoch dans l'ordre rng.permutation, clipping puis AdamW à chaque mini-batch, loss pesée par "
                   "le nombre d'exemples.")

### Les réponses de ta feuille

**CP4.1 — Questions flash**

In [ ]:
# CP4.1: the values written on your answer sheet
answer_CP4_1a = ...  # a) number of parameters
answer_CP4_1b = ...  # b) a shape, as a string like "(1, 2)"
answer_CP4_1c = ...  # c) a list of four values
answer_CP4_1d = ...  # d) a probability, 3 decimals
answer_CP4_1e = ...  # e) number of learned parameters
answer_CP4_1f = ...  # f) number of parameters

if exam_over():
    for letter, answer in zip("abcdef", [answer_CP4_1a, as_shape(answer_CP4_1b), answer_CP4_1c, answer_CP4_1d, answer_CP4_1e, answer_CP4_1f]):
        RESULTS[f"CP4.1{letter}"] = wb.check(f"CP4.1{letter}", answer)

**CP4.2 — Passe avant, rétropropagation et un pas de SGD**

In [ ]:
# CP4.2: the values written on your answer sheet
answer_CP4_2a = ...  # a) [z1, z2]
answer_CP4_2b = ...  # b) [y_hat, L], 5 decimals
answer_CP4_2c = ...  # c) delta = dL/dy_hat
answer_CP4_2d = ...  # d) [dW2_1, dW2_2, db2], 4 decimals
answer_CP4_2e = ...  # e) [dL/dz1, dL/dz2], 4 decimals
answer_CP4_2f = ...  # f) dL/dW1 as a list of two rows, 4 decimals
answer_CP4_2g = ...  # g) new W1 as a list of two rows, 5 decimals
answer_CP4_2h = ...  # h) loss after the step, 5 decimals

if exam_over():
    for letter, answer in zip("abcdefgh", [answer_CP4_2a, answer_CP4_2b, answer_CP4_2c, answer_CP4_2d, answer_CP4_2e, answer_CP4_2f, answer_CP4_2g, answer_CP4_2h]):
        RESULTS[f"CP4.2{letter}"] = wb.check(f"CP4.2{letter}", answer)

**CP4.3 — Momentum, Adam et AdamW**

In [ ]:
# CP4.3: the values written on your answer sheet
answer_CP4_3a = ...  # a) b2
answer_CP4_3b = ...  # b) w2
answer_CP4_3c = ...  # c) w1, 5 decimals
answer_CP4_3d = ...  # d) [m_hat_2, v_hat_2], 3 decimals
answer_CP4_3e = ...  # e) w2, 5 decimals
answer_CP4_3f = ...  # f) w1 with AdamW, 5 decimals
answer_CP4_3g = ...  # g) w1 with Adam(weight_decay=0.1), 5 decimals

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP4_3a, answer_CP4_3b, answer_CP4_3c, answer_CP4_3d, answer_CP4_3e, answer_CP4_3f, answer_CP4_3g]):
        RESULTS[f"CP4.3{letter}"] = wb.check(f"CP4.3{letter}", answer)

**CP4.4 — Dropout et batchnorm**

In [ ]:
# CP4.4: the values written on your answer sheet
answer_CP4_4a = ...  # a) a list, 3 decimals
answer_CP4_4b = ...  # b) a list
answer_CP4_4c = ...  # c) the mean of a)
answer_CP4_4d = ...  # d) [mean, biased variance]
answer_CP4_4e = ...  # e) output for 12, 4 decimals
answer_CP4_4f = ...  # f) [running mean, running var], 4 decimals
answer_CP4_4g = ...  # g) eval output for 7, 4 decimals

if exam_over():
    for letter, answer in zip("abcdefg", [answer_CP4_4a, answer_CP4_4b, answer_CP4_4c, answer_CP4_4d, answer_CP4_4e, answer_CP4_4f, answer_CP4_4g]):
        RESULTS[f"CP4.4{letter}"] = wb.check(f"CP4.4{letter}", answer)

**CP4.5 — Le gradient à travers des sigmoïdes empilées (c)**

In [ ]:
# CP4.5: the values written on your answer sheet
answer_CP4_5c = ...  # c) the smallest number of layers

if exam_over():
    for letter, answer in zip("c", [answer_CP4_5c]):
        RESULTS[f"CP4.5{letter}"] = wb.check(f"CP4.5{letter}", answer)

**CP4.6 — Fermi : paramètres et mémoire**

In [ ]:
# CP4.6: the values written on your answer sheet
answer_CP4_6a = ...  # a) parameters in millions, 1 decimal
answer_CP4_6b = ...  # b) memory in MB (an integer)
answer_CP4_6c = ...  # c) the factor
answer_CP4_6d = ...  # d) number of updates

if exam_over():
    for letter, answer in zip("abcd", [answer_CP4_6a, answer_CP4_6b, answer_CP4_6c, answer_CP4_6d]):
        RESULTS[f"CP4.6{letter}"] = wb.check(f"CP4.6{letter}", answer)

**CP4.7 — Les lettres des quatre lignes fautives (a)**

In [ ]:
# CP4.7: the values written on your answer sheet
answer_CP4_7a = ...  # a) four letters, for example "XYZW"

if exam_over():
    for letter, answer in zip("a", [as_sorted_letters(answer_CP4_7a)]):
        RESULTS[f"CP4.7{letter}"] = wb.check(f"CP4.7{letter}", answer)

**CP4.8 — Lire les quatre paires de courbes**

In [ ]:
# CP4.8: the values written on your answer sheet
answer_CP4_8a = ...  # a) a letter A to F
answer_CP4_8b = ...  # b) a letter A to F
answer_CP4_8c = ...  # c) a letter A to F
answer_CP4_8d = ...  # d) a letter A to F

if exam_over():
    for letter, answer in zip("abcd", [as_letters(answer_CP4_8a), as_letters(answer_CP4_8b), as_letters(answer_CP4_8c), as_letters(answer_CP4_8d)]):
        RESULTS[f"CP4.8{letter}"] = wb.check(f"CP4.8{letter}", answer)

**CP4.12 — Parties I à III**

In [ ]:
# CP4.12: the values written on your answer sheet
answer_CP4_12a = ...  # a) log loss in nats, 3 decimals
answer_CP4_12b = ...  # b) log loss in bits, 3 decimals
answer_CP4_12c = ...  # c) mean log loss, 4 decimals
answer_CP4_12d = ...  # d) [w for alpha = 0, w for alpha = 7], 3 decimals
answer_CP4_12e = ...  # e) z-score, 3 decimals

if exam_over():
    for letter, answer in zip("abcde", [answer_CP4_12a, answer_CP4_12b, answer_CP4_12c, answer_CP4_12d, answer_CP4_12e]):
        RESULTS[f"CP4.12{letter}"] = wb.check(f"CP4.12{letter}", answer)

**CP4.13 — Prédire ce que font cinq extraits PyTorch**

In [ ]:
# CP4.13: the values written on your answer sheet
answer_CP4_13a = ...  # a) a letter A, B, C or D
answer_CP4_13b = ...  # b) a letter A, B, C or D
answer_CP4_13c = ...  # c) a letter A, B, C or D
answer_CP4_13d = ...  # d) a letter A, B, C or D
answer_CP4_13e = ...  # e) a letter A, B, C or D

if exam_over():
    for letter, answer in zip("abcde", [as_letters(answer_CP4_13a), as_letters(answer_CP4_13b), as_letters(answer_CP4_13c), as_letters(answer_CP4_13d), as_letters(answer_CP4_13e)]):
        RESULTS[f"CP4.13{letter}"] = wb.check(f"CP4.13{letter}", answer)

### Bilan

In [ ]:
# Summary of the automatic checks
import re


def natural(sub_id):
    """CP4.2a before CP4.13a: the question number as a number, then the letter."""
    number, letter = re.match(r"CP4\.(\d+)(.*)", sub_id).groups()
    return int(number), letter


if exam_over():
    done = {k: r for k, r in RESULTS.items() if r.status != "pending"}
    right = sorted((k for k, r in done.items() if r), key=natural)
    wrong = sorted((k for k, r in done.items() if not r), key=natural)
    pending = sorted((k for k, r in RESULTS.items() if r.status == "pending"), key=natural)
    print(f"Réponses vérifiées : {len(right)} juste(s) sur {len(done)} ; pas remplies : {len(pending)}.")
    if wrong:
        print("À revoir avec le corrigé :", ", ".join(wrong))
    checked = [ok for props in PROPERTIES.values() for _, ok in props]
    print(f"Propriétés du code (CP4.9) : {sum(checked)} sur {len(checked)}.")
    print("Note ensuite ta copie avec le barème de 03_examen_corrige.md : il donne aussi les points des démarches, "
          "des justifications et des questions rédigées, que cette vérification ne voit pas.")

**Ce que cette vérification ne voit pas** : les démarches et les justifications ; les explications et les démonstrations (CP4.2 i, CP4.3 h, CP4.4 c (le pourquoi) et h, CP4.5 a, b et d, CP4.7 b, CP4.10, CP4.11, CP4.12 f, CP4.14) ; et, pour CP4.9, les nombres recopiés sur ta feuille (seules tes fonctions sont vérifiées). Tout cela se corrige avec `03_examen_corrige.md`. Ensuite : la remédiation du corrigé (relis la synthèse, `05_synthese.md`, pour les notions ratées), puis le mini-projet.